# Notebook 7 — Prototype Methods Comparison (NearestNeighbors vs FAISS)

## Purpose
This notebook compares two prototype-search methods for the Diagnostic Agent prototype branch:

- **NearestNeighbors**
- **FAISS**

## Current prototype setting
- real data only
- current benchmark classes:
  - `RC_CAPACITY_OVERLOAD`
  - `RC_TRANSPORT_DELAY`
- main temporal benchmark window:
  - **10 samples**
- best temporal model from Notebook 6:
  - **GRU Autoencoder**

This notebook is designed to produce:
- prototype-classification metrics
- confusion matrices
- retrieval-quality comparison
- class prototype visualizations
- presentation-ready Plotly figures
- a clear winner for the prototype branch

## Why this notebook comes now

From Notebook 6, the **GRU Autoencoder** was selected as the best temporal model because it achieved:
- lower validation reconstruction loss than LSTM
- lower mean test reconstruction error than LSTM
- and it was selected as the best temporal model in the saved summary

So this notebook uses **GRU latent embeddings** as the main prototype space and compares:
- exact nearest-neighbor search with scikit-learn
- fast vector search with FAISS

In [1]:
import os
import gc
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

VERBOSE = True
SAVE_ARTIFACTS = True
EXPORT_PLOTS = True
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

def log(msg, level="INFO"):
    if VERBOSE:
        print(f"[{level}] {msg}")

def banner(title):
    if VERBOSE:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)

def timed_run(name, fn, *args, **kwargs):
    start = time.time()
    log(f"START: {name}")
    result = fn(*args, **kwargs)
    elapsed = time.time() - start
    log(f"END: {name} | elapsed = {elapsed:.2f} sec")
    return result

banner("NOTEBOOK 7 INITIALIZATION")
log("Notebook 7 initialized")


NOTEBOOK 7 INITIALIZATION
[INFO] Notebook 7 initialized


In [2]:
banner("INSTALL / IMPORT VISUALIZATION PACKAGES")

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    import sys
    !{sys.executable} -m pip install -q plotly
    import plotly.express as px
    import plotly.graph_objects as go

try:
    import kaleido  # noqa: F401
    log("kaleido already available")
except Exception:
    log("Installing kaleido for Plotly PNG export...", "WARN")
    import sys
    !{sys.executable} -m pip install -q kaleido
    import kaleido  # noqa: F401
    log("kaleido installed successfully")

def save_plot(fig, name):
    if not EXPORT_PLOTS:
        return
    fig.write_html(f"{name}.html")
    try:
        fig.write_image(f"{name}.png", scale=2)
        log(f"Saved {name}.html and {name}.png")
    except Exception as e:
        log(f"Saved {name}.html | PNG export skipped ({e})", "WARN")


INSTALL / IMPORT VISUALIZATION PACKAGES
[WARN] Installing kaleido for Plotly PNG export...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.9 MB/s eta 0:00:00
[INFO] kaleido installed successfully


## Device compatibility note

Some Kaggle accelerator environments report CUDA as available, but the installed PyTorch build may not support the exact GPU architecture for recurrent layers like **GRU/LSTM**.

This safe notebook performs a **real recurrent-layer CUDA compatibility test** before selecting the device.
If that test fails, it automatically falls back to **CPU**.

In [3]:
banner("INSTALL / IMPORT MODELING PACKAGES")

# Torch
try:
    import torch
    import torch.nn as nn
except Exception:
    log("torch not found. Installing now...", "WARN")
    import sys
    !{sys.executable} -m pip install -q torch
    import torch
    import torch.nn as nn

# sklearn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
import joblib

# FAISS
try:
    import faiss
    FAISS_OK = True
except Exception:
    log("faiss not found. Installing faiss-cpu...", "WARN")
    import sys
    !{sys.executable} -m pip install -q faiss-cpu
    import faiss
    FAISS_OK = True
    log("faiss installed successfully")

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

def select_safe_rnn_device():
    """Choose CUDA only if it is actually usable for recurrent models, else fall back to CPU."""
    if not torch.cuda.is_available():
        return "cpu", "CUDA not available"

    try:
        # Basic CUDA tensor test
        _ = torch.tensor([1.0], device="cuda")

        # Recurrent-model compatibility test
        test_model = nn.GRU(input_size=4, hidden_size=4, batch_first=True).to("cuda")
        test_x = torch.randn(2, 3, 4, device="cuda")
        with torch.no_grad():
            _ = test_model(test_x)

        del test_model, test_x, _
        torch.cuda.empty_cache()
        return "cuda", "CUDA recurrent compatibility test passed"
    except Exception as e:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return "cpu", f"CUDA unusable for recurrent layers in this runtime: {e}"

DEVICE, DEVICE_REASON = select_safe_rnn_device()
log(f"Using device: {DEVICE}")
log(f"Device decision reason: {DEVICE_REASON}")

if DEVICE == "cpu":
    log("Falling back to CPU is acceptable here because the benchmark latent extraction workload is still relatively small.", "WARN")


INSTALL / IMPORT MODELING PACKAGES
[WARN] faiss not found. Installing faiss-cpu...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.6 MB/s eta 0:00:00:00:0100:01
[INFO] faiss installed successfully
[INFO] Using device: cpu
[INFO] Device decision reason: CUDA unusable for recurrent layers in this runtime: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

[WARN] Falling back to CPU is acceptable here because the benchmark latent extraction workload is still relatively small.


## Load Notebook 6 summary

We verify:
- selected temporal model
- target classes
- benchmark window size

In [4]:
banner("LOAD NOTEBOOK 6 SUMMARY")

with open("/kaggle/input/notebooks/azizamriiiiiiiiii/06-temporal-models-compare/nb6_temporal_model_summary.json", "r") as f:
    nb6_summary = json.load(f)

display(pd.DataFrame([nb6_summary]))

WINDOW_SIZE = nb6_summary["window_size"]
TARGET_CLASSES = nb6_summary["target_classes"]
BEST_TEMPORAL_MODEL = nb6_summary["best_temporal_model"]

log(f"Window size = {WINDOW_SIZE}")
log(f"Target classes = {TARGET_CLASSES}")
log(f"Best temporal model = {BEST_TEMPORAL_MODEL}")


LOAD NOTEBOOK 6 SUMMARY


,window_size,target_classes,lstm_best_val_loss,lstm_best_epoch,lstm_train_time_sec,gru_best_val_loss,gru_best_epoch,gru_train_time_sec,lstm_test_recon_mean,gru_test_recon_mean,lstm_test_latent_silhouette,gru_test_latent_silhouette,best_temporal_model
0,10,"[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY]",0.401665,30,1.738099,0.390505,29,2.640879,0.508348,0.454893,0.031914,0.013141,GRU


[INFO] Window size = 10
[INFO] Target classes = ['RC_CAPACITY_OVERLOAD', 'RC_TRANSPORT_DELAY']
[INFO] Best temporal model = GRU


## Load sequence artifacts and scaler

We reconstruct the latent embeddings from:
- the saved sequence splits from Notebook 3
- the saved scaler from Notebook 6
- the saved GRU model weights

In [5]:
banner("LOAD SEQUENCE DATA AND SCALER")

train_npz = np.load(f"/kaggle/input/notebooks/azizamriiiiiiiiii/03-feature-window-split-build/seq_w{WINDOW_SIZE}_train.npz", allow_pickle=True)
val_npz   = np.load(f"/kaggle/input/notebooks/azizamriiiiiiiiii/03-feature-window-split-build/seq_w{WINDOW_SIZE}_val.npz", allow_pickle=True)
test_npz  = np.load(f"/kaggle/input/notebooks/azizamriiiiiiiiii/03-feature-window-split-build/seq_w{WINDOW_SIZE}_test.npz", allow_pickle=True)

X_train = train_npz["X"]
y_train_raw = train_npz["y"]

X_val = val_npz["X"]
y_val_raw = val_npz["y"]

X_test = test_npz["X"]
y_test_raw = test_npz["y"]

scaler = joblib.load("/kaggle/input/notebooks/azizamriiiiiiiiii/06-temporal-models-compare/nb6_sequence_scaler.joblib")

log(f"Raw X_train shape = {X_train.shape}")
log(f"Raw X_val shape   = {X_val.shape}")
log(f"Raw X_test shape  = {X_test.shape}")


LOAD SEQUENCE DATA AND SCALER
[INFO] Raw X_train shape = (354, 10, 16)
[INFO] Raw X_val shape   = (58, 10, 16)
[INFO] Raw X_test shape  = (68, 10, 16)


In [6]:
banner("STANDARDIZE SEQUENCE FEATURES")

n_features = X_train.shape[-1]

X_train_scaled = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape).astype("float32")
X_val_scaled   = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape).astype("float32")
X_test_scaled  = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape).astype("float32")

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

class_names = list(le.classes_)
log(f"Class names = {class_names}")


STANDARDIZE SEQUENCE FEATURES
[INFO] Class names = [np.str_('RC_CAPACITY_OVERLOAD'), np.str_('RC_TRANSPORT_DELAY')]


## Rebuild the GRU autoencoder and load weights

This notebook uses the GRU temporal model selected in Notebook 6.

In [7]:
banner("DEFINE GRU AUTOENCODER")

class BaseSeqAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

    def encode(self, x):
        raise NotImplementedError

    def reconstruct(self, x):
        raise NotImplementedError

    def forward(self, x):
        return self.reconstruct(x)

class GRUAutoencoder(BaseSeqAutoencoder):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32):
        super().__init__(input_dim, hidden_dim, latent_dim)
        self.encoder = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        _, h_n = self.encoder(x)
        h = h_n[-1]
        z = self.to_latent(h)
        return z

    def reconstruct(self, x):
        seq_len = x.size(1)
        z = self.encode(x)
        dec_init = self.from_latent(z)
        dec_seq = dec_init.unsqueeze(1).repeat(1, seq_len, 1)
        dec_out, _ = self.decoder(dec_seq)
        out = self.output_layer(dec_out)
        return out

INPUT_DIM = X_train_scaled.shape[-1]
HIDDEN_DIM = 64
LATENT_DIM = 32

gru_ae = GRUAutoencoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM).to(DEVICE)
state_dict = torch.load("/kaggle/input/notebooks/azizamriiiiiiiiii/06-temporal-models-compare/nb6_gru_autoencoder.pt", map_location=DEVICE)
gru_ae.load_state_dict(state_dict)
gru_ae.eval()

log("Loaded GRU autoencoder weights successfully")


DEFINE GRU AUTOENCODER
[INFO] Loaded GRU autoencoder weights successfully


## Extract latent embeddings

We compute train / validation / test latent embeddings using the GRU encoder.

In [8]:
banner("EXTRACT GRU LATENT EMBEDDINGS")

def get_latent_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        x_tensor = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        z = model.encode(x_tensor).cpu().numpy()
    return z

Z_train = timed_run("extract train latents", get_latent_embeddings, gru_ae, X_train_scaled)
Z_val   = timed_run("extract val latents", get_latent_embeddings, gru_ae, X_val_scaled)
Z_test  = timed_run("extract test latents", get_latent_embeddings, gru_ae, X_test_scaled)

log(f"Z_train shape = {Z_train.shape}")
log(f"Z_val shape   = {Z_val.shape}")
log(f"Z_test shape  = {Z_test.shape}")


EXTRACT GRU LATENT EMBEDDINGS
[INFO] START: extract train latents
[INFO] END: extract train latents | elapsed = 0.08 sec
[INFO] START: extract val latents
[INFO] END: extract val latents | elapsed = 0.00 sec
[INFO] START: extract test latents
[INFO] END: extract test latents | elapsed = 0.00 sec
[INFO] Z_train shape = (354, 32)
[INFO] Z_val shape   = (58, 32)
[INFO] Z_test shape  = (68, 32)


## Build class prototypes

For each class, we compute the centroid in latent space.

These class centroids are useful both for:
- interpretation
- visualization
- prototype diagnostics

In [9]:
banner("BUILD CLASS PROTOTYPES")

prototype_rows = []
prototype_vectors = {}

for class_name in class_names:
    mask = (y_train_raw == class_name)
    class_z = Z_train[mask]
    centroid = class_z.mean(axis=0)
    prototype_vectors[class_name] = centroid
    prototype_rows.append({
        "label": class_name,
        "n_train_windows": int(mask.sum())
    })

prototype_df = pd.DataFrame(prototype_rows)
display(prototype_df)

prototype_matrix = np.vstack([prototype_vectors[c] for c in class_names]).astype("float32")


BUILD CLASS PROTOTYPES


,label,n_train_windows
0,RC_CAPACITY_OVERLOAD,208
1,RC_TRANSPORT_DELAY,146


## Prototype visualization with PCA

This figure is useful for the presentation because it shows:
- train windows in latent space
- class centroids as prototypes

In [10]:
banner("PLOT LATENT SPACE WITH PROTOTYPES")

pca = PCA(n_components=2, random_state=RANDOM_SEED)
Z_train_2d = pca.fit_transform(Z_train)
proto_2d = pca.transform(prototype_matrix)

latent_plot_df = pd.DataFrame({
    "pc1": Z_train_2d[:, 0],
    "pc2": Z_train_2d[:, 1],
    "label": y_train_raw,
    "kind": "train_window"
})

proto_plot_df = pd.DataFrame({
    "pc1": proto_2d[:, 0],
    "pc2": proto_2d[:, 1],
    "label": class_names,
    "kind": "prototype"
})

fig_proto = go.Figure()

for label in class_names:
    df_sub = latent_plot_df[latent_plot_df["label"] == label]
    fig_proto.add_trace(go.Scatter(
        x=df_sub["pc1"],
        y=df_sub["pc2"],
        mode="markers",
        name=f"{label} train windows",
        opacity=0.55
    ))

for _, row in proto_plot_df.iterrows():
    fig_proto.add_trace(go.Scatter(
        x=[row["pc1"]],
        y=[row["pc2"]],
        mode="markers+text",
        name=f'{row["label"]} prototype',
        text=[row["label"]],
        textposition="top center",
        marker=dict(size=16, symbol="x")
    ))

fig_proto.update_layout(
    title="GRU Latent Space with Class Prototypes (PCA)",
    xaxis_title="PC1",
    yaxis_title="PC2",
    template="plotly_white"
)
fig_proto.show()

save_plot(fig_proto, "nb7_gru_latent_prototypes_pca")


PLOT LATENT SPACE WITH PROTOTYPES


[WARN] Saved nb7_gru_latent_prototypes_pca.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Define prototype-prediction helpers

We compare two prototype-search methods:
- **NearestNeighbors**
- **FAISS**

In [11]:
banner("DEFINE PROTOTYPE HELPERS")

def compute_metrics(y_true, y_pred, split_name, method_name):
    return {
        "method": method_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

def plot_confusion_matrix(cm, labels, title):
    fig = px.imshow(
        cm,
        x=labels,
        y=labels,
        text_auto=True,
        color_continuous_scale="Blues",
        title=title
    )
    fig.update_layout(template="plotly_white", xaxis_title="Predicted", yaxis_title="Actual")
    return fig

def topk_hit_rate(y_true, neighbor_labels, k):
    hits = 0
    for i, true_label in enumerate(y_true):
        if true_label in neighbor_labels[i, :k]:
            hits += 1
    return hits / len(y_true)

def majority_vote(labels_row):
    vals, counts = np.unique(labels_row, return_counts=True)
    return vals[np.argmax(counts)]


DEFINE PROTOTYPE HELPERS


## Method 1 — NearestNeighbors

We use the GRU train latent space as the prototype/search database.

In [12]:
banner("NEARESTNEIGHBORS SEARCH")

K = 5

nn_start = time.time()
nn_index = NearestNeighbors(n_neighbors=K, metric="euclidean")
nn_index.fit(Z_train)
nn_fit_time = time.time() - nn_start

# Validation search
nn_val_start = time.time()
nn_val_dist, nn_val_idx = nn_index.kneighbors(Z_val, n_neighbors=K)
nn_val_time = time.time() - nn_val_start

# Test search
nn_test_start = time.time()
nn_test_dist, nn_test_idx = nn_index.kneighbors(Z_test, n_neighbors=K)
nn_test_time = time.time() - nn_test_start

nn_val_neighbor_labels = y_train_raw[nn_val_idx]
nn_test_neighbor_labels = y_train_raw[nn_test_idx]

nn_val_pred = np.array([majority_vote(row) for row in nn_val_neighbor_labels])
nn_test_pred = np.array([majority_vote(row) for row in nn_test_neighbor_labels])

log(f"NN fit time = {nn_fit_time:.4f} sec")
log(f"NN val search time = {nn_val_time:.4f} sec")
log(f"NN test search time = {nn_test_time:.4f} sec")


NEARESTNEIGHBORS SEARCH
[INFO] NN fit time = 0.0008 sec
[INFO] NN val search time = 0.0469 sec
[INFO] NN test search time = 0.0012 sec


## Method 2 — FAISS

We use the same train latent space, but with FAISS vector indexing.

In [13]:
banner("FAISS SEARCH")

Z_train_f32 = Z_train.astype("float32")
Z_val_f32   = Z_val.astype("float32")
Z_test_f32  = Z_test.astype("float32")

faiss_start = time.time()
faiss_index = faiss.IndexFlatL2(Z_train_f32.shape[1])
faiss_index.add(Z_train_f32)
faiss_fit_time = time.time() - faiss_start

# Validation search
faiss_val_start = time.time()
faiss_val_dist, faiss_val_idx = faiss_index.search(Z_val_f32, K)
faiss_val_time = time.time() - faiss_val_start

# Test search
faiss_test_start = time.time()
faiss_test_dist, faiss_test_idx = faiss_index.search(Z_test_f32, K)
faiss_test_time = time.time() - faiss_test_start

faiss_val_neighbor_labels = y_train_raw[faiss_val_idx]
faiss_test_neighbor_labels = y_train_raw[faiss_test_idx]

faiss_val_pred = np.array([majority_vote(row) for row in faiss_val_neighbor_labels])
faiss_test_pred = np.array([majority_vote(row) for row in faiss_test_neighbor_labels])

log(f"FAISS fit time = {faiss_fit_time:.4f} sec")
log(f"FAISS val search time = {faiss_val_time:.4f} sec")
log(f"FAISS test search time = {faiss_test_time:.4f} sec")


FAISS SEARCH
[INFO] FAISS fit time = 0.0002 sec
[INFO] FAISS val search time = 0.0040 sec
[INFO] FAISS test search time = 0.0028 sec


## Evaluate prototype methods

We evaluate both methods on:
- validation set
- test set

We also compute:
- top-1 classification quality
- top-3 hit rate
- top-5 hit rate
- search time

In [14]:
banner("COMPUTE PROTOTYPE METRICS")

rows = []

# NearestNeighbors metrics
rows.append({
    **compute_metrics(y_val_raw, nn_val_pred, "val", "NearestNeighbors"),
    "top3_hit_rate": topk_hit_rate(y_val_raw, nn_val_neighbor_labels, 3),
    "top5_hit_rate": topk_hit_rate(y_val_raw, nn_val_neighbor_labels, 5),
    "search_time_sec": nn_val_time
})
rows.append({
    **compute_metrics(y_test_raw, nn_test_pred, "test", "NearestNeighbors"),
    "top3_hit_rate": topk_hit_rate(y_test_raw, nn_test_neighbor_labels, 3),
    "top5_hit_rate": topk_hit_rate(y_test_raw, nn_test_neighbor_labels, 5),
    "search_time_sec": nn_test_time
})

# FAISS metrics
rows.append({
    **compute_metrics(y_val_raw, faiss_val_pred, "val", "FAISS"),
    "top3_hit_rate": topk_hit_rate(y_val_raw, faiss_val_neighbor_labels, 3),
    "top5_hit_rate": topk_hit_rate(y_val_raw, faiss_val_neighbor_labels, 5),
    "search_time_sec": faiss_val_time
})
rows.append({
    **compute_metrics(y_test_raw, faiss_test_pred, "test", "FAISS"),
    "top3_hit_rate": topk_hit_rate(y_test_raw, faiss_test_neighbor_labels, 3),
    "top5_hit_rate": topk_hit_rate(y_test_raw, faiss_test_neighbor_labels, 5),
    "search_time_sec": faiss_test_time
})

proto_metrics_df = pd.DataFrame(rows)
display(proto_metrics_df)

proto_metrics_df.to_csv("nb7_prototype_metrics.csv", index=False)
log("Saved nb7_prototype_metrics.csv")


COMPUTE PROTOTYPE METRICS


,method,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,top3_hit_rate,top5_hit_rate,search_time_sec
0,NearestNeighbors,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.000000,0.046858
1,NearestNeighbors,test,0.632353,0.632353,0.593593,0.593593,0.713986,0.632353,0.779412,0.897059,0.001198
2,FAISS,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.000000,0.003958
3,FAISS,test,0.632353,0.632353,0.593593,0.593593,0.713986,0.632353,0.779412,0.897059,0.002849


[INFO] Saved nb7_prototype_metrics.csv


## Plot validation and test metric comparison

This figure is useful for the presentation because it compares:
- classification performance
- retrieval quality
- search speed

In [15]:
banner("PLOT PROTOTYPE METRIC COMPARISON")

plot_metrics = [
    "accuracy", "balanced_accuracy", "macro_f1",
    "top3_hit_rate", "top5_hit_rate", "search_time_sec"
]

proto_long = proto_metrics_df.melt(
    id_vars=["method", "split"],
    value_vars=plot_metrics,
    var_name="metric",
    value_name="value"
)

fig_proto_metrics = px.bar(
    proto_long,
    x="metric",
    y="value",
    color="method",
    barmode="group",
    facet_col="split",
    title="Prototype Method Comparison Across Validation and Test Metrics",
    text="value"
)
fig_proto_metrics.update_traces(texttemplate="%{text:.3f}")
fig_proto_metrics.update_layout(template="plotly_white")
fig_proto_metrics.show()

save_plot(fig_proto_metrics, "nb7_prototype_method_comparison")


PLOT PROTOTYPE METRIC COMPARISON


[WARN] Saved nb7_prototype_method_comparison.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Confusion matrices on the test set

These confusion matrices show how the two prototype methods classify the benchmark classes.

In [16]:
banner("TEST CONFUSION MATRICES")

nn_cm_test = confusion_matrix(y_test_raw, nn_test_pred, labels=class_names)
faiss_cm_test = confusion_matrix(y_test_raw, faiss_test_pred, labels=class_names)

fig_nn_cm = plot_confusion_matrix(nn_cm_test, class_names, "NearestNeighbors — Test Confusion Matrix")
fig_faiss_cm = plot_confusion_matrix(faiss_cm_test, class_names, "FAISS — Test Confusion Matrix")

fig_nn_cm.show()
fig_faiss_cm.show()

save_plot(fig_nn_cm, "nb7_nn_test_confusion_matrix")
save_plot(fig_faiss_cm, "nb7_faiss_test_confusion_matrix")


TEST CONFUSION MATRICES


[WARN] Saved nb7_nn_test_confusion_matrix.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)
[WARN] Saved nb7_faiss_test_confusion_matrix.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Prototype confidence proxy

We use inverse-distance style confidence proxies to compare how sharply each method retrieves its nearest matches.

In [17]:
banner("PLOT PROTOTYPE CONFIDENCE PROXY")

# Simple confidence proxy from nearest distance
nn_conf = 1.0 / (1.0 + nn_test_dist[:, 0])
faiss_conf = 1.0 / (1.0 + faiss_test_dist[:, 0])

conf_df = pd.DataFrame({
    "confidence_proxy": np.concatenate([nn_conf, faiss_conf]),
    "method": (["NearestNeighbors"] * len(nn_conf)) + (["FAISS"] * len(faiss_conf))
})

fig_conf = px.histogram(
    conf_df,
    x="confidence_proxy",
    color="method",
    barmode="overlay",
    nbins=30,
    title="Prototype Confidence Proxy Distribution — Test Set"
)
fig_conf.update_layout(template="plotly_white")
fig_conf.show()

save_plot(fig_conf, "nb7_prototype_confidence_proxy")


PLOT PROTOTYPE CONFIDENCE PROXY


[WARN] Saved nb7_prototype_confidence_proxy.html | PNG export skipped (
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
)


## Example retrieval cases

We display a few test examples and the labels of their nearest retrieved neighbors.
This is useful qualitative evidence for the presentation.

In [18]:
banner("EXAMPLE RETRIEVAL CASES")

example_count = min(10, len(y_test_raw))

example_rows = []
for i in range(example_count):
    example_rows.append({
        "true_label": y_test_raw[i],
        "nn_pred": nn_test_pred[i],
        "faiss_pred": faiss_test_pred[i],
        "nn_neighbors": list(nn_test_neighbor_labels[i]),
        "faiss_neighbors": list(faiss_test_neighbor_labels[i]),
        "nn_nearest_dist": float(nn_test_dist[i, 0]),
        "faiss_nearest_dist": float(faiss_test_dist[i, 0]),
    })

example_df = pd.DataFrame(example_rows)
display(example_df)

example_df.to_csv("nb7_example_retrieval_cases.csv", index=False)
log("Saved nb7_example_retrieval_cases.csv")


EXAMPLE RETRIEVAL CASES


,true_label,nn_pred,faiss_pred,nn_neighbors,faiss_neighbors,nn_nearest_dist,faiss_nearest_dist
0,RC_CAPACITY_OVERLOAD,RC_TRANSPORT_DELAY,RC_TRANSPORT_DELAY,"[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...","[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...",0.301652,0.090993
1,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.220774,0.048741
2,RC_TRANSPORT_DELAY,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.440665,0.194186
3,RC_TRANSPORT_DELAY,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.894472,0.800080
4,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.730373,0.533446
5,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.929724,0.864387
6,RC_TRANSPORT_DELAY,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.818571,0.670057
7,RC_TRANSPORT_DELAY,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...","[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...",0.801627,0.642606
8,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...","[RC_CAPACITY_OVERLOAD, RC_CAPACITY_OVERLOAD, R...",0.722017,0.521308
9,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,RC_CAPACITY_OVERLOAD,"[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...","[RC_CAPACITY_OVERLOAD, RC_TRANSPORT_DELAY, RC_...",0.866061,0.750062


[INFO] Saved nb7_example_retrieval_cases.csv


## Select the best prototype method

Selection rule:
1. best validation Macro F1
2. if close, better validation top-3 hit rate
3. if still close, faster search time

In [19]:
banner("SELECT BEST PROTOTYPE METHOD")

val_rank = proto_metrics_df[proto_metrics_df["split"] == "val"].copy()
val_rank = val_rank.sort_values(
    by=["macro_f1", "top3_hit_rate", "search_time_sec"],
    ascending=[False, False, True]
).reset_index(drop=True)

best_method = val_rank.iloc[0]["method"]
log(f"Best prototype method selected = {best_method}")

best_test_metrics = proto_metrics_df[
    (proto_metrics_df["split"] == "test") &
    (proto_metrics_df["method"] == best_method)
].copy()

display(val_rank)
display(best_test_metrics)


SELECT BEST PROTOTYPE METHOD
[INFO] Best prototype method selected = FAISS


,method,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,top3_hit_rate,top5_hit_rate,search_time_sec
0,FAISS,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.0,0.003958
1,NearestNeighbors,val,0.775862,0.779762,0.774184,0.773513,0.791925,0.779762,0.948276,1.0,0.046858


,method,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,top3_hit_rate,top5_hit_rate,search_time_sec
3,FAISS,test,0.632353,0.632353,0.593593,0.593593,0.713986,0.632353,0.779412,0.897059,0.002849


## Save artifacts and summary

We save:
- metrics
- qualitative retrieval examples
- selected winner
- prototype vectors

In [20]:
banner("SAVE NOTEBOOK 7 OUTPUTS")

np.savez_compressed(
    "nb7_class_prototypes.npz",
    prototype_matrix=prototype_matrix,
    class_names=np.array(class_names, dtype=object)
)

summary = {
    "window_size": WINDOW_SIZE,
    "target_classes": class_names,
    "best_temporal_model_used": BEST_TEMPORAL_MODEL,
    "best_prototype_method": best_method,
    "nn_fit_time_sec": float(nn_fit_time),
    "nn_val_search_time_sec": float(nn_val_time),
    "nn_test_search_time_sec": float(nn_test_time),
    "faiss_fit_time_sec": float(faiss_fit_time),
    "faiss_val_search_time_sec": float(faiss_val_time),
    "faiss_test_search_time_sec": float(faiss_test_time),
    "validation_metrics": val_rank.to_dict(orient="records"),
    "best_test_metrics": best_test_metrics.to_dict(orient="records")
}

with open("nb7_prototype_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

log("Saved Notebook 7 artifacts")


SAVE NOTEBOOK 7 OUTPUTS
[INFO] Saved Notebook 7 artifacts
